# Milestone 8 — Legal Comparison

Notebook mở rộng pipeline M5–M7 để đối chiếu Luật Đất đai 2013 với pháp luật hiện hành. Hybrid Retriever, context builder, Qwen 4-bit và deterministic citation validation được **tái sử dụng**, không rebuild index và không để LLM tự tạo citation.

## 1. Mount Google Drive và vào repo

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path('/content/drive/MyDrive/Master/IT2043/Project/Final_Project/vn-landlaw-qa')
assert PROJECT_ROOT.exists(), f'Không tìm thấy repo: {PROJECT_ROOT}'
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print('Project root:', PROJECT_ROOT)

Project root: /content/drive/MyDrive/Master/IT2043/Project/Final_Project/vn-landlaw-qa


## 2. Cài dependency và kiểm tra Colab T4

In [3]:
%pip install -q -r requirements/generation.txt

In [4]:
import torch
assert torch.cuda.is_available(), 'Hãy chọn Runtime > Change runtime type > T4 GPU'
gpu = torch.cuda.get_device_properties(0)
print(f'GPU: {gpu.name} | VRAM: {gpu.total_memory / 1024**3:.2f} GB')

GPU: Tesla T4 | VRAM: 14.56 GB


## 3. Load artifacts và pipeline M5 — không rebuild

BM25 và FAISS được load từ Drive. M8 chỉ gọi lại `HybridRetriever.search()`.

In [5]:
import json

CHUNKS_PATH = PROJECT_ROOT / 'data/processed/chunks.json'
BM25_INDEX_PATH = PROJECT_ROOT / 'indexes/bm25/bm25_index.json.gz'
FAISS_DIR = PROJECT_ROOT / 'indexes/faiss'
REQUIRED = [
    CHUNKS_PATH, BM25_INDEX_PATH, FAISS_DIR / 'faiss.index',
    FAISS_DIR / 'vector_metadata.json', FAISS_DIR / 'embedding_manifest.json',
]
missing = [str(path) for path in REQUIRED if not path.exists()]
assert not missing, 'Thiếu artifacts M3–M5:\n' + '\n'.join(missing)
all_chunks = json.loads(CHUNKS_PATH.read_text(encoding='utf-8'))
print(f'Chunks: {len(all_chunks):,}; artifacts sẵn sàng.')

Chunks: 921; artifacts sẵn sàng.


In [6]:
from landlaw_rag.retrieval import (
    BM25Index, VectorIndex, BM25Retriever, VectorRetriever,
    HybridConfig, HybridRetriever,
)

bm25_index = BM25Index.load(BM25_INDEX_PATH)
# Chỉ encode query trên CPU Colab để dành toàn bộ VRAM T4 cho Qwen.
# FAISS index đã được build sẵn; notebook không encode lại 921 chunks.
vector_index = VectorIndex.load(
    FAISS_DIR, device='cpu', load_encoder=True, chunks_path=CHUNKS_PATH
)
hybrid_retriever = HybridRetriever(
    BM25Retriever(bm25_index),
    VectorRetriever(vector_index),
    HybridConfig(bm25_top_n=20, vector_top_n=20, final_top_k=6, rrf_k=60),
)
print(f'BM25: {len(bm25_index.documents):,} | FAISS: {vector_index.index.ntotal:,}')
print('Embedding query device: CPU Colab | Generation device: T4 GPU')

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BM25: 921 | FAISS: 921
Embedding query device: CPU Colab | Generation device: T4 GPU


## 4. Load Qwen2.5-7B-Instruct 4-bit trên T4

Dùng cùng checkpoint và cấu hình NF4 của M6. Model được cache trên Google Drive.

In [7]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

MAX_CONTEXT_TOKENS_PER_SIDE = 2600 #2600
MAX_INPUT_TOKENS = 7000 #7000
MAX_NEW_TOKENS = 512 #512
TOP_K_PER_SIDE = 4

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

torch.cuda.empty_cache()

print("Loading model from Hugging Face:", MODEL_NAME)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

model.eval()

model.generation_config.do_sample = False
model.generation_config.temperature = None
model.generation_config.top_p = None
model.generation_config.top_k = None

print("Model ready:", MODEL_NAME)
print(f"Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"Reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

Loading model from Hugging Face: Qwen/Qwen2.5-7B-Instruct


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Model ready: Qwen/Qwen2.5-7B-Instruct
Allocated: 5.18 GB
Reserved:  5.33 GB


## 5. Import API M6–M8

- M6: context builder và article expansion.
- M7: `cite_tagged_answer()` sinh citation deterministic từ metadata.
- M8: intent routing, hai lượt retrieval, version isolation và evidence mapping.

In [8]:
from landlaw_rag.generation import (
    CitationValidationError, build_context, cite_tagged_answer,
    expand_article_context, format_source,
)
from landlaw_rag.comparison import (
    MISSING_COUNTERPART_MESSAGE, build_comparison_messages,
    build_validation_retry_messages,
    canonicalize_comparison_structure,
    detect_legal_intent, offset_source_labels,
    retrieve_comparison_evidence, validate_comparison_mapping,
    validate_comparison_structure,
)

ROUTING_EXAMPLES = [
    'Điều kiện chuyển nhượng quyền sử dụng đất là gì?',
    'Theo Luật Đất đai 2013, điều kiện chuyển nhượng là gì?',
    'So sánh điều kiện chuyển nhượng theo luật 2013 và 2024.',
    'Quy định trước đây và hiện nay khác nhau thế nào?',
]
for question in ROUTING_EXAMPLES:
    print(detect_legal_intent(question), '->', question)

qa -> Điều kiện chuyển nhượng quyền sử dụng đất là gì?
qa -> Theo Luật Đất đai 2013, điều kiện chuyển nhượng là gì?
comparison -> So sánh điều kiện chuyển nhượng theo luật 2013 và 2024.
comparison -> Quy định trước đây và hiện nay khác nhau thế nào?


## 6. Retrieve riêng hai phía và in evidence để debug

Hai truy vấn dùng cùng chủ đề pháp lý nhưng gắn định danh phiên bản khác nhau. Sau retrieval có hard-filter theo năm trong metadata; số Điều không được dùng để map hai luật.

In [9]:
def _print_evidence(label: str, items: list[dict] | tuple[dict, ...]) -> None:
    print(f'\n{label} ({len(items)} chunks)')
    for item in items:
        score = item.get('final_score')
        score_text = f'{score:.6f}' if score is not None else 'expanded'
        focus = item.get('legal_focus_score')
        focus_text = f'{focus:.6f}' if focus is not None else 'n/a'
        print(
            f"- {item.get('chunk_id')} | {format_source(item.get('metadata', {}))} "
            f"| hybrid={score_text} | legal_focus={focus_text}"
        )

def retrieve_and_prepare(query: str, top_k_per_side: int = TOP_K_PER_SIDE, debug: bool = True):
    separated = retrieve_comparison_evidence(
        hybrid_retriever, query, top_k_per_side=top_k_per_side, candidate_multiplier=4
    )
    legacy = expand_article_context(
        list(separated.legacy), all_chunks, MAX_CONTEXT_TOKENS_PER_SIDE, tokenizer
    )
    current = expand_article_context(
        list(separated.current), all_chunks, MAX_CONTEXT_TOKENS_PER_SIDE, tokenizer
    )
    if debug:
        print('Chủ đề chung:', separated.topic)
        print('Query 2013:', separated.legacy_query)
        print('Query hiện hành:', separated.current_query)
        _print_evidence('EVIDENCE 2013', legacy)
        _print_evidence('EVIDENCE HIỆN HÀNH', current)
    return separated, legacy, current

## 7. Sinh structured comparison và validate citation M7

Nhãn `[Nguồn n]` chỉ là vị trí evidence trong context. Sau generation, M7 kiểm tra nhãn và tạo citation pháp lý từ metadata. M8 kiểm tra thêm claim từng mục có trỏ đúng phía evidence hay không.

In [10]:
import gc
import time

@torch.inference_mode()
def legal_comparison_answer(
    query: str, top_k_per_side: int = TOP_K_PER_SIDE,
    max_new_tokens: int = MAX_NEW_TOKENS, debug: bool = True,
    max_validation_retries: int = 2,
) -> dict:
    if detect_legal_intent(query) != 'comparison':
        return {'mode': 'qa', 'query': query, 'message': 'Chuyển sang pipeline QA M6.'}

    total_start = time.perf_counter()
    retrieval_start = time.perf_counter()
    separated, legacy, current = retrieve_and_prepare(query, top_k_per_side, debug)
    retrieval_seconds = time.perf_counter() - retrieval_start

    if not legacy or not current:
        return {
            'mode': 'comparison', 'query': query,
            'answer': MISSING_COUNTERPART_MESSAGE,
            'legacy_sources': legacy, 'current_sources': current,
            'citations': [], 'claim_citation_mapping': [],
            'retrieval_seconds': retrieval_seconds,
            'generation_seconds': 0.0,
            'total_seconds': time.perf_counter() - total_start,
        }

    legacy_context, legacy_used = build_context(
        legacy, tokenizer, max_context_tokens=MAX_CONTEXT_TOKENS_PER_SIDE
    )
    current_context, current_used = build_context(
        current, tokenizer, max_context_tokens=MAX_CONTEXT_TOKENS_PER_SIDE
    )
    current_context = offset_source_labels(current_context, len(legacy_used))
    evidence = legacy_used + current_used
    validation_errors = []
    generated_parts = []
    input_token_counts = []
    output_token_counts = []
    generation_start = time.perf_counter()
    for sections in ((1, 2), (3, 4)):
        base_messages = build_comparison_messages(
            query, legacy_context, current_context, sections=sections
        )
        messages = base_messages
        for attempt in range(max_validation_retries + 1):
            prompt = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            inputs = tokenizer(prompt, return_tensors='pt', truncation=False)
            input_count = int(inputs['input_ids'].shape[1])
            if input_count > MAX_INPUT_TOKENS:
                raise ValueError(
                    f'Prompt có {input_count} tokens, vượt MAX_INPUT_TOKENS={MAX_INPUT_TOKENS}. '
                    'Cần điều chỉnh context/prompt; không cắt âm thầm evidence.'
                )
            inputs = inputs.to(model.device)
            torch.cuda.synchronize()
            generated = model.generate(
                **inputs, max_new_tokens=min(max_new_tokens, MAX_NEW_TOKENS),
                do_sample=False, repetition_penalty=1.05,
                pad_token_id=tokenizer.eos_token_id,
            )
            torch.cuda.synchronize()
            output_tokens = generated[0, input_count:]
            output_count = int(output_tokens.shape[0])
            raw_part = tokenizer.decode(output_tokens, skip_special_tokens=True).strip()
            raw_part = canonicalize_comparison_structure(raw_part)
            del inputs, generated, output_tokens
            gc.collect()
            torch.cuda.empty_cache()
            try:
                expected = (
                    '1. Quy định năm 2013:', '2. Quy định hiện hành:'
                ) if sections == (1, 2) else (
                    '3. Điểm giống:', '4. Điểm khác/thay đổi:'
                )
                lines = [line.strip() for line in raw_part.splitlines() if line.strip()]
                if len(lines) != 2 or any(
                    not line.startswith(heading)
                    for line, heading in zip(lines, expected)
                ):
                    raise CitationValidationError(
                        f'Mục {sections} thiếu hoặc sai cấu trúc; output={raw_part!r}'
                    )
                partial = cite_tagged_answer(raw_part, evidence)
                validate_comparison_mapping(
                    partial.claim_mappings, legacy_used, current_used
                )
                generated_parts.append(raw_part)
                input_token_counts.append(input_count)
                output_token_counts.append(output_count)
                break
            except CitationValidationError as exc:
                validation_errors.append(str(exc))
                if output_count >= min(max_new_tokens, MAX_NEW_TOKENS):
                    raise CitationValidationError(
                        f'Mục {sections} chạm giới hạn {output_count} output tokens; '
                        f'câu trả lời bị cắt. Output: {raw_part!r}'
                    ) from exc
                if attempt >= max_validation_retries:
                    raise
                if debug:
                    print(f'Validation retry mục {sections}: {exc}')
                messages = build_validation_retry_messages(
                    base_messages, raw_part, sections
                )
    raw_answer = '\n'.join(generated_parts)
    validate_comparison_structure(raw_answer)
    cited = cite_tagged_answer(raw_answer, evidence)
    validate_comparison_mapping(cited.claim_mappings, legacy_used, current_used)
    generation_seconds = time.perf_counter() - generation_start

    return {
        'mode': 'comparison', 'query': query, 'answer': cited.text,
        'legacy_sources': legacy_used, 'current_sources': current_used,
        'citations': [
            {'number': c.number, 'text': c.text, 'chunk_ids': list(c.chunk_ids)}
            for c in cited.citations
        ],
        'claim_citation_mapping': [
            {'claim': m.claim, 'citation_ids': list(m.citation_ids), 'chunk_ids': list(m.chunk_ids)}
            for m in cited.claim_mappings
        ],
        'input_tokens': input_token_counts,
        'output_tokens': output_token_counts,
        'validation_attempts': len(validation_errors) + 2,
        'validation_errors': validation_errors,
        'retrieval_seconds': retrieval_seconds,
        'generation_seconds': generation_seconds,
        'total_seconds': time.perf_counter() - total_start,
    }

In [11]:
def print_comparison_result(result: dict) -> None:
    print('MODE:', result['mode'])
    print('CÂU HỎI:', result['query'])
    print('\nFINAL COMPARISON + VALIDATED CITATIONS:\n')
    print(result.get('answer') or result.get('message'))
    if result.get('claim_citation_mapping'):
        print('\nCLAIM → CHUNK MAPPING:')
        for mapping in result['claim_citation_mapping']:
            print('-', mapping)
    print(
        f"\nTiming: retrieval={result.get('retrieval_seconds', 0):.3f}s | "
        f"generation={result.get('generation_seconds', 0):.3f}s | "
        f"total={result.get('total_seconds', 0):.3f}s"
    )

## 8. Demo câu hỏi so sánh 2013–2024

In [12]:
DEMO_QUESTIONS = [
    # 'So sánh điều kiện để cá nhân được chuyển nhượng quyền sử dụng đất theo Luật Đất đai 2013 và pháp luật hiện hành.',
    # 'So sánh quy định bồi thường về đất khi Nhà nước thu hồi đất giữa năm 2013 và hiện nay.',
    # 'So sánh điều kiện để người sử dụng đất được chuyển nhượng quyền sử dụng đất theo Luật Đất đai 2013 và Luật Đất đai 2024.?',
    'So sánh nghĩa vụ chung của người sử dụng đất theo Luật Đất đai 2013 và Luật Đất đai 2024.?',
]

demo_results = []
for question in DEMO_QUESTIONS:
    result = legal_comparison_answer(question, debug=True)
    demo_results.append(result)
    print_comparison_result(result)
    print('\n' + '=' * 100 + '\n')

Chủ đề chung: nghĩa vụ chung của người sử dụng đất
Query 2013: nghĩa vụ chung của người sử dụng đất Luật Đất đai 2013 45/2013/QH13
Query hiện hành: nghĩa vụ chung của người sử dụng đất Luật Đất đai 2024 31/2024/QH15 hiện hành

EVIDENCE 2013 (4 chunks)
- 45-2013-qh13-34b35a6d8317da14 | Luật Đất đai số 45/2013/QH13 → Chương XI - QUYỀN VÀ NGHĨA VỤ CỦA NGƯỜI SỬ DỤNG ĐẤT → Mục 1 - QUY ĐỊNH CHUNG → Điều 170. Nghĩa vụ chung của người sử dụng đất | hybrid=0.050835 | legal_focus=12.000000
- 45-2013-qh13-3647c47e689aa73b | Luật Đất đai số 45/2013/QH13 → Chương XI - QUYỀN VÀ NGHĨA VỤ CỦA NGƯỜI SỬ DỤNG ĐẤT → Mục 4 - QUYỀN VÀ NGHĨA VỤ CỦA NGƯỜI VIỆT ĐỊNH CƯ Ở NƯỚC NGOÀI, TỔ CHỨC NƯỜC NGOÀI CÓ CHỨC NĂNG NGOẠI GIAO, DOANH NGHIỆP CÓ VỐN ĐẦU TƯ NƯỚC NGOÀI SỬ DỤNG ĐẤT → Điều 183. Quyền và nghĩa vụ của người Việt Nam định cư ở nước ngoài, doanh nghiệp có vốn đầu tư nước ngoài sử dụng đất để thực hiện dự án đầu tư tại Việt Nam → Khoản 1 | hybrid=0.051025 | legal_focus=11.500000
- 45-2013-qh13-ba524538a8dc

## 9. Kiểm tra tiêu chí hoàn thành M8

Cell cuối chạy unit test và kiểm tra invariants của các demo: routing, hai pool không giao nhau, citation thuộc evidence và đủ bốn mục.

In [13]:
!PYTHONPATH=src python -m pytest tests/test_legal_comparison.py tests/test_hybrid.py tests/test_rag_generation.py tests/test_citation.py -q

assert all(detect_legal_intent(q) == 'comparison' for q in DEMO_QUESTIONS)
regression = demo_results[0]
assert regression['legacy_sources'][0]['metadata']['article'] == 'Điều 188'
assert regression['current_sources'][0]['metadata']['article'] == 'Điều 45'
assert regression['current_sources'][0]['metadata']['clause'] == 'Khoản 1'
for result in demo_results:
    legacy_ids = {item['chunk_id'] for item in result['legacy_sources']}
    current_ids = {item['chunk_id'] for item in result['current_sources']}
    assert legacy_ids.isdisjoint(current_ids), 'Evidence hai phiên bản bị trộn'
    if result['answer'] != MISSING_COUNTERPART_MESSAGE:
        assert all(
            heading in result['answer']
            for heading in (
                '1. Quy định năm 2013:', '2. Quy định hiện hành:',
                '3. Điểm giống:', '4. Điểm khác/thay đổi:',
            )
        )
        cited_ids = {cid for c in result['citations'] for cid in c['chunk_ids']}
        assert cited_ids <= legacy_ids | current_ids

print('M8 PASS: routing, separate retrieval, version isolation, structured output,')
print('evidence mapping và deterministic citation validation đều đạt.')

...........................................................              [100%]
59 passed in 0.26s


AssertionError: 